# Baseline Preprocessing
This notebook creates cleaned baseline CSVs in `./PreProcessdData` by removing URL-like columns, fixing data types, cleaning categorical values, handling missing data, and capping extreme outliers. IDs are kept for joins.

In [11]:
import os
import re
import pandas as pd

DATA_DIR = './Data'
OUTPUT_DIR = './PreProcessdData'

datasets = {
    'player_profiles': f'{DATA_DIR}/player_profiles/player_profiles.csv',
    'transfer_history': f'{DATA_DIR}/transfer_history/transfer_history.csv',
    'team_details': f'{DATA_DIR}/team_details/team_details.csv',
    'team_competitions_seasons': f'{DATA_DIR}/team_competitions_seasons/team_competitions_seasons.csv',
    'player_performances': f'{DATA_DIR}/player_performances/player_performances.csv',
    'player_national_performances': f'{DATA_DIR}/player_national_performances/player_national_performances.csv',
    'player_market_value': f'{DATA_DIR}/player_market_value/player_market_value.csv',
    'player_latest_market_value': f'{DATA_DIR}/player_latest_market_value/player_latest_market_value.csv',
    'player_injuries': f'{DATA_DIR}/player_injuries/player_injuries.csv',
    'player_teammates_played_with': f'{DATA_DIR}/player_teammates_played_with/player_teammates_played_with.csv',
    'team_children': f'{DATA_DIR}/team_children/team_children.csv',
}

drop_keywords = ['url', 'href', 'link', 'path', 'image']
date_keywords = ['date', 'day', 'dob', 'from', 'to', 'since', 'until', 'start', 'end']
id_keywords = ['id']

missing_drop_threshold = 0.85
winsor_low = 0.01
winsor_high = 0.99

critical_columns = {
    
}

colmuns_not_to_drop = {
    'player_profiles': ['outfitter', 'contract_option', 'contract_there_expires', 'second_club_name']
}

columns_to_drop = {
    'player_profiles': ['date_of_death'],
}

def is_drop_column(col_name):
    lower = col_name.lower()
    return any(keyword in lower for keyword in drop_keywords)

def is_id_column(col_name):
    lower = col_name.lower()
    return any(keyword == lower or lower.endswith(keyword) for keyword in id_keywords)

def is_date_column(col_name):
    lower = col_name.lower()
    return any(keyword in lower for keyword in date_keywords)

def clean_string_series(series):
    return (
        series.astype(str)
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
    )

def coerce_numeric(series):
    cleaned = (
        series.astype(str)
        .str.replace(',', '', regex=False)
        .str.replace(r'[^0-9.+-]', '', regex=True)
    )
    return pd.to_numeric(cleaned, errors='coerce')

def winsorize_series(series, low_q=winsor_low, high_q=winsor_high):
    if not pd.api.types.is_numeric_dtype(series) or pd.api.types.is_bool_dtype(series):
        return series
    data = series.dropna()
    if data.empty:
        return series
    low = data.quantile(low_q)
    high = data.quantile(high_q)
    if low == high:
        return series
    return series.clip(lower=low, upper=high)

def preprocess_table(name, path):
    df = pd.read_csv(path)


    # Drop URL-like and non-informative columns and to drop colmuns specified in columns_to_drop
    drop_cols = [c for c in df.columns if is_drop_column(c) or c in columns_to_drop.get(name, [])]

    df = df.drop(columns=drop_cols, errors='ignore')

    # Fix data types: numeric strings and dates
    for col in df.columns:
        if df[col].dtype == 'object':
            if is_date_column(col):
                df[col] = pd.to_datetime(df[col], errors='coerce', infer_datetime_format=True)
            else:
                numeric_candidate = coerce_numeric(df[col])
                numeric_ratio = numeric_candidate.notna().mean()
                if numeric_ratio >= 0.7:
                    df[col] = numeric_candidate
                else:
                    df[col] = clean_string_series(df[col])

    # Missingness flags (only for columns with missing values)
    for col in df.columns:
        if df[col].isna().any():
            df[f'{col}_was_missing'] = df[col].isna().astype(int)

    # Drop high-missingness columns (except IDs and dates and columns in colmuns_not_to_drop)
    miss_rate = df.isna().mean()
    drop_missing = [
        c for c in df.columns
        if miss_rate.get(c, 0) >= missing_drop_threshold
        and not is_id_column(c)
        and not is_date_column(c)
        and c not in colmuns_not_to_drop.get(name, [])
    ]
    df = df.drop(columns=drop_missing, errors='ignore')

    # Impute missing values (skip IDs and dates)
    critical = set(critical_columns.get(name, []))
    for col in df.columns:
        if col in critical or (df[col].isna().any() and not is_id_column(col) and not is_date_column(col)):
            if pd.api.types.is_numeric_dtype(df[col]):
                fill_value = df[col].median()
                df[col] = df[col].fillna(fill_value)
            elif df[col].dtype == 'object':
                mode = df[col].mode(dropna=True)
                if not mode.empty:
                    df[col] = df[col].fillna(mode.iloc[0])

    # Winsorize numeric columns (skip IDs and booleans)
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]) and not pd.api.types.is_bool_dtype(df[col]) and not is_id_column(col):
            df[col] = winsorize_series(df[col])

    output_dir = os.path.join(OUTPUT_DIR, name)
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, f'{name}.csv')
    df.to_csv(output_path, index=False)

    print(f'{name}: saved -> {output_path}')
    print(f'  Dropped URL-like columns ({len(drop_cols)}): {drop_cols}')
    print(f'  Dropped high-missing columns ({len(drop_missing)}): {drop_missing}')

for name, path in datasets.items():
    preprocess_table(name, path)

C:\Users\User\AppData\Local\Temp\ipykernel_28936\2044401112.py:82: DtypeWarning: Columns (0: third_club_url, 1: third_club_name, 2: fourth_club_url, 3: fourth_club_name) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


player_profiles: saved -> ./PreProcessdData\player_profiles\player_profiles.csv
  Dropped URL-like columns (6): ['player_image_url', 'social_media_url', 'second_club_url', 'third_club_url', 'fourth_club_url', 'date_of_death']
  Dropped high-missing columns (2): ['third_club_name', 'fourth_club_name']
transfer_history: saved -> ./PreProcessdData\transfer_history\transfer_history.csv
  Dropped URL-like columns (0): []
  Dropped high-missing columns (0): []
team_details: saved -> ./PreProcessdData\team_details\team_details.csv
  Dropped URL-like columns (2): ['logo_url', 'source_url']
  Dropped high-missing columns (0): []
team_competitions_seasons: saved -> ./PreProcessdData\team_competitions_seasons\team_competitions_seasons.csv
  Dropped URL-like columns (0): []
  Dropped high-missing columns (0): []
player_performances: saved -> ./PreProcessdData\player_performances\player_performances.csv
  Dropped URL-like columns (0): []
  Dropped high-missing columns (0): []
player_national_perfor